# BirdCLEF 2026 | 4-Model Public Blend

Ravi版 Public Blend V6 (no overlap/TTA) + 自分のeffnetv2_b3 v2 ONNX 5-fold + eca_nfnet_l0 SED ONNX 5-fold

**Models:**
1. LB872.pt - Tony Li finetuned EfficientNet-B0 SED (~8min)
2. LB862.pt - AidenSong baseline EfficientNet-B0 SED (~8min)
3. effnetv2_b3 v2 ONNX 5-fold - 自作モデル (~35min)
4. eca_nfnet_l0 SED ONNX 5-fold - 自作モデル (~35min)

**Target: ~90min total (90min limit)**

In [ ]:
!pip install /kaggle/input/onnxruntime-wheel/onnxruntime-*.whl -q --no-deps 2>/dev/null || pip install /kaggle/input/datasets/ttyn4519/onnxruntime-wheel/onnxruntime-*.whl -q --no-deps

In [ ]:
import os
print("=== /kaggle/input/ contents ===")
if os.path.exists("/kaggle/input"):
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  {d}/")
        subpath = f"/kaggle/input/{d}"
        if os.path.isdir(subpath):
            for f in sorted(os.listdir(subpath))[:10]:
                print(f"    {f}")
else:
    print("  /kaggle/input does not exist")

## Model 1: LB 0.872 (Tony Li finetuned)

In [ ]:
%%writefile lb872.py

# ============================================================
# BirdCLEF 2026 | Model 1 Inference
# MODEL: LB872.pt  (finetuned, EfficientNet-B0 SED, val_auc ~0.996)
# OUTPUT: lb872.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "lb872.csv"

@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4
    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()

DATA_ROOT    = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR     = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
CKPT         = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB872.pt"
assert os.path.exists(CKPT), f"Missing checkpoint: {CKPT}"

class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {"clipwise_logit": clipwise_logit, "clipwise_prob": torch.sigmoid(clipwise_logit)}

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
            features_only=False, global_pool="", num_classes=0, drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x): return self.head(self.gem_pool(self.backbone(x)))

class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax, power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.db(torch.nan_to_num(self.mel(waveforms))), nan=-80.0, posinf=0.0, neginf=-80.0)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        return torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0).unsqueeze(1).repeat(1, 3, 1, 1)

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)

ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model = SEDModel(cfg)
model.load_state_dict(state, strict=True)
model.eval()
mel_transform = MelSpectrogramTransform(cfg).eval()
print(f"LB872 (finetuned): loaded")

row_pattern = re.compile(r"^(.*)_(\d+)$")
def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem: expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem: expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
if not test_files:
    for stem in sorted(expected_by_stem):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p): test_files.append(p)
    print(f"No test .ogg found; using {len(test_files)} train_soundscapes fallback files.")
else:
    print(f"Found {len(test_files)} test soundscape files.")

def load_soundscape(path):
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(y, nan=0.0).astype(np.float32), Path(path).stem

print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
    results = list(ex.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time()-t0:.1f}s")

CHUNK = cfg.chunk_samples
all_row_ids, all_preds = [], []

print("Running inference...")
t0 = time.time()
with torch.no_grad():
    for audio, stem in results:
        n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration) if stem in expected_by_stem else max(1, len(audio) // CHUNK)
        padded_len = n_chunks * CHUNK
        audio = np.pad(audio, (0, max(0, padded_len - len(audio))))[:padded_len]
        chunks_tensor = torch.from_numpy(audio.reshape(n_chunks, CHUNK)).float()
        probs = model(mel_transform(chunks_tensor))["clipwise_prob"].clamp(0.0, 1.0).cpu().numpy()
        if stem in expected_by_stem:
            valid = [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))
        all_row_ids.extend([f"{stem}_{(i+1)*int(cfg.chunk_duration)}" for i in valid])
        all_preds.extend(probs[valid])

print(f"Inference done in {time.time()-t0:.1f}s")

# -- SAVE (handle empty predictions for Save & Run mode) --
if len(all_preds) > 0:
    pred_df = pd.DataFrame(np.stack(all_preds), columns=SPECIES, index=pd.Index(all_row_ids, name="row_id"))
    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]
else:
    pred_df = pd.DataFrame(columns=SPECIES, dtype=np.float32)
    pred_df.index.name = "row_id"

missing = [rid for rid in expected_ids if rid not in pred_df.index]
if missing:
    pred_df = pd.concat([pred_df, pd.DataFrame(np.zeros((len(missing), len(SPECIES)), dtype=np.float32),
        columns=SPECIES, index=pd.Index(missing, name="row_id"))])
pred_df = pred_df.loc[expected_ids]
pred_df.to_csv(SUBMISSION_FILE, index=True)
print(f"Saved {SUBMISSION_FILE} - shape = {pred_df.shape}\n")

## Model 2: LB 0.862 (AidenSong baseline)

In [ ]:
%%writefile lb862.py

# ============================================================
# BirdCLEF 2026 | Model 2 Inference (same arch as LB872, different weights)
# MODEL: LB862.pt  (baseline, EfficientNet-B0 SED, val_auc ~0.948)
# OUTPUT: lb862.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "lb862.csv"

@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    max_workers: int = 4
    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()

DATA_ROOT    = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR     = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
CKPT         = "/kaggle/input/datasets/tonylica/birdclef-2026-model/LB862.pt"
assert os.path.exists(CKPT), f"Missing checkpoint: {CKPT}"

class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {"clipwise_logit": clipwise_logit, "clipwise_prob": torch.sigmoid(clipwise_logit)}

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg.backbone, pretrained=False, in_chans=cfg.in_channels,
            features_only=False, global_pool="", num_classes=0, drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x): return self.head(self.gem_pool(self.backbone(x)))

class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax, power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = torch.nan_to_num(self.db(torch.nan_to_num(self.mel(waveforms))), nan=-80.0, posinf=0.0, neginf=-80.0)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        return torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0).unsqueeze(1).repeat(1, 3, 1, 1)

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES = list(sample_sub.columns[1:])
cfg.num_classes = len(SPECIES)

ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model = SEDModel(cfg)
model.load_state_dict(state, strict=True)
model.eval()
mel_transform = MelSpectrogramTransform(cfg).eval()
print(f"LB862 (baseline): loaded")

row_pattern = re.compile(r"^(.*)_(\d+)$")
def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem: expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem: expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
if not test_files:
    for stem in sorted(expected_by_stem):
        p = os.path.join(TRAIN_SC_DIR, f"{stem}.ogg")
        if os.path.exists(p): test_files.append(p)
    print(f"No test .ogg found; using {len(test_files)} train_soundscapes fallback files.")
else:
    print(f"Found {len(test_files)} test soundscape files.")

def load_soundscape(path):
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(y, nan=0.0).astype(np.float32), Path(path).stem

print("Loading audio...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
    results = list(ex.map(load_soundscape, test_files))
print(f"Loaded {len(results)} files in {time.time()-t0:.1f}s")

CHUNK = cfg.chunk_samples
all_row_ids, all_preds = [], []

print("Running inference...")
t0 = time.time()
with torch.no_grad():
    for audio, stem in results:
        n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration) if stem in expected_by_stem else max(1, len(audio) // CHUNK)
        padded_len = n_chunks * CHUNK
        audio = np.pad(audio, (0, max(0, padded_len - len(audio))))[:padded_len]
        chunks_tensor = torch.from_numpy(audio.reshape(n_chunks, CHUNK)).float()
        probs = model(mel_transform(chunks_tensor))["clipwise_prob"].clamp(0.0, 1.0).cpu().numpy()
        if stem in expected_by_stem:
            valid = [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))
        all_row_ids.extend([f"{stem}_{(i+1)*int(cfg.chunk_duration)}" for i in valid])
        all_preds.extend(probs[valid])

print(f"Inference done in {time.time()-t0:.1f}s")

if len(all_preds) > 0:
    pred_df = pd.DataFrame(np.stack(all_preds), columns=SPECIES, index=pd.Index(all_row_ids, name="row_id"))
    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]
else:
    pred_df = pd.DataFrame(columns=SPECIES, dtype=np.float32)
    pred_df.index.name = "row_id"

missing = [rid for rid in expected_ids if rid not in pred_df.index]
if missing:
    pred_df = pd.concat([pred_df, pd.DataFrame(np.zeros((len(missing), len(SPECIES)), dtype=np.float32),
        columns=SPECIES, index=pd.Index(missing, name="row_id"))])
pred_df = pred_df.loc[expected_ids]
pred_df.to_csv(SUBMISSION_FILE, index=True)
print(f"Saved {SUBMISSION_FILE} - shape = {pred_df.shape}\n")

## Model 3: effnetv2_b3 v2 ONNX 5-fold (LB 0.886)

In [ ]:
%%writefile effnetv2_b3.py

# ============================================================
# BirdCLEF 2026 | Model 3 Inference
# MODEL: SED EfficientNetV2-B3 v2 ONNX 5-fold (LB 0.886)
# OUTPUT: effnetv2_b3.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf
import torch
import torchaudio

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "effnetv2_b3.csv"

SR = 32000
DURATION = 5
N_SAMPLES = SR * DURATION
N_FFT = 2048
N_MELS = 224
F_MIN = 0
F_MAX = 16000
HOP_LENGTH = 512
MEL_NORM = "slaney"
MEL_SCALE = "htk"
NUM_CLASSES = 234

DATA_ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_ROOT, "sample_submission.csv")

# Find model directory (path varies between direct dataset and nested datasets)
MODEL_DIR_CANDIDATES = [
    "/kaggle/input/sed-effnetv2-b3-models",
    "/kaggle/input/datasets/ttyn4519/sed-effnetv2-b3-models",
]
MODEL_DIR = None
for d in MODEL_DIR_CANDIDATES:
    if os.path.isdir(d):
        MODEL_DIR = d
        break
assert MODEL_DIR is not None, f"Model dir not found in: {MODEL_DIR_CANDIDATES}"
print(f"Model dir: {MODEL_DIR}")

ONNX_PATHS = [f"{MODEL_DIR}/sed_effnetv2_b3_fold{fold}_best.onnx" for fold in range(5)]

def create_ort_session(onnx_path):
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 4
    sess_options.inter_op_num_threads = 1
    return ort.InferenceSession(onnx_path, sess_options, providers=["CPUExecutionProvider"])

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
    norm=MEL_NORM, mel_scale=MEL_SCALE,
)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

@torch.no_grad()
def compute_mel(waveforms):
    mel = mel_transform(waveforms)
    mel = amp_to_db(mel)
    B = mel.shape[0]
    mel_flat = mel.reshape(B, -1)
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
    return mel.unsqueeze(1).repeat(1, 3, 1, 1).numpy()

def main():
    t_total_start = time.time()

    sub_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    target_columns = [c for c in sub_df.columns if c != "row_id"]
    print(f"Target columns: {len(target_columns)} classes")

    row_ids = sub_df["row_id"].values
    end_seconds = np.array([int(rid.rsplit("_", 1)[1]) for rid in row_ids])
    filenames = np.array([rid.rsplit("_", 1)[0] + ".ogg" for rid in row_ids])

    unique_files = list(dict.fromkeys(filenames))
    print(f"Unique soundscape files: {len(unique_files)}")

    existing_files = [f for f in unique_files if os.path.isfile(f"{TEST_DIR}/{f}")]
    if not existing_files:
        existing_files = [f for f in unique_files if os.path.isfile(f"{TRAIN_SC_DIR}/{f}")]
        soundscape_dir = TRAIN_SC_DIR
        print(f"No test .ogg found; using {len(existing_files)} train_soundscapes fallback files.")
    else:
        soundscape_dir = TEST_DIR
        if len(existing_files) < len(unique_files):
            print(f"  Available test files: {len(existing_files)}/{len(unique_files)}")
    unique_files = existing_files

    # Load ONNX models (skip if no files to process - Save & Run mode)
    sessions = []
    if len(unique_files) > 0:
        for fold_idx, onnx_path in enumerate(ONNX_PATHS):
            print(f"Loading ONNX model fold {fold_idx}: {onnx_path}")
            sessions.append(create_ort_session(onnx_path))
        input_name = sessions[0].get_inputs()[0].name
        print(f"  ONNX input name: {input_name}")
    else:
        print("No files to process, skipping model loading.")

    row_id_to_idx = {rid: i for i, rid in enumerate(row_ids)}
    predictions = sub_df[target_columns].values.astype(np.float32).copy()

    n_processed_files = 0
    n_processed_segments = 0
    t_inference_start = time.time()

    for file_idx, fname in enumerate(unique_files):
        filepath = f"{soundscape_dir}/{fname}"
        file_mask = filenames == fname
        file_end_seconds = end_seconds[file_mask]
        file_row_ids = row_ids[file_mask]

        audio, file_sr = sf.read(filepath, dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        segments = []
        for end_sec in file_end_seconds:
            start_sample = (end_sec - DURATION) * SR
            end_sample = end_sec * SR
            if start_sample < 0:
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[:max(end_sample, 0)]
                if len(available) > 0: seg[N_SAMPLES - len(available):] = available
            elif end_sample > len(audio):
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[min(start_sample, len(audio)):]
                if len(available) > 0: seg[:len(available)] = available
            else:
                seg = audio[start_sample:end_sample]
            segments.append(seg)

        mel_batch = compute_mel(torch.from_numpy(np.stack(segments)).float())

        clipwise_probs_sum = np.zeros((len(segments), NUM_CLASSES), dtype=np.float32)
        for session in sessions:
            for i in range(mel_batch.shape[0]):
                outputs = session.run(None, {input_name: mel_batch[i:i+1]})
                clipwise_probs_sum[i] += outputs[0][0]
        clipwise_probs = clipwise_probs_sum / len(sessions)

        for i, rid in enumerate(file_row_ids):
            predictions[row_id_to_idx[rid]] = clipwise_probs[i]

        n_processed_files += 1
        n_processed_segments += len(file_end_seconds)
        if n_processed_files % 100 == 0 or n_processed_files == len(unique_files):
            elapsed = time.time() - t_inference_start
            speed = n_processed_segments / elapsed if elapsed > 0 else 0
            print(f"  [{n_processed_files}/{len(unique_files)}] {n_processed_segments} segments, {elapsed:.1f}s, {speed:.1f} seg/s")

    sub_df[target_columns] = predictions
    sub_df.set_index("row_id").to_csv(SUBMISSION_FILE)

    t_total = time.time() - t_total_start
    print(f"\nInference complete: {n_processed_files} files, {n_processed_segments} segments, {t_total:.1f}s ({t_total/60:.1f}min)")
    print(f"Output: {SUBMISSION_FILE}")

if __name__ == "__main__":
    main()

## Model 4: eca_nfnet_l0 SED 5-fold ONNX (LB 0.885)

In [ ]:
%%writefile eca_nfnet_l0.py

# ============================================================
# BirdCLEF 2026 | Model 4 Inference
# MODEL: SED eca_nfnet_l0 ONNX 5-fold (LB 0.885)
# OUTPUT: eca_nfnet_l0.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf
import torch
import torchaudio

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "eca_nfnet_l0.csv"

SR = 32000
DURATION = 5
N_SAMPLES = SR * DURATION
N_FFT = 2048
N_MELS = 224
F_MIN = 0
F_MAX = 16000
HOP_LENGTH = 512
MEL_NORM = "slaney"
MEL_SCALE = "htk"
NUM_CLASSES = 234

DATA_ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_ROOT, "sample_submission.csv")

# Find model directory (path varies between direct dataset and nested datasets)
MODEL_DIR_CANDIDATES = [
    "/kaggle/input/sed-eca-nfnet-l0-models",
    "/kaggle/input/datasets/ttyn4519/sed-eca-nfnet-l0-models",
]
MODEL_DIR = None
for d in MODEL_DIR_CANDIDATES:
    if os.path.isdir(d):
        MODEL_DIR = d
        break
assert MODEL_DIR is not None, f"Model dir not found in: {MODEL_DIR_CANDIDATES}"
print(f"Model dir: {MODEL_DIR}")

ONNX_PATHS = [f"{MODEL_DIR}/sed_eca_nfnet_l0_fold{fold}_best.onnx" for fold in range(5)]

def create_ort_session(onnx_path):
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 4
    sess_options.inter_op_num_threads = 1
    return ort.InferenceSession(onnx_path, sess_options, providers=["CPUExecutionProvider"])

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
    norm=MEL_NORM, mel_scale=MEL_SCALE,
)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

@torch.no_grad()
def compute_mel(waveforms):
    mel = mel_transform(waveforms)
    mel = amp_to_db(mel)
    B = mel.shape[0]
    mel_flat = mel.reshape(B, -1)
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
    return mel.unsqueeze(1).repeat(1, 3, 1, 1).numpy()

def main():
    t_total_start = time.time()

    sub_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    target_columns = [c for c in sub_df.columns if c != "row_id"]
    print(f"Target columns: {len(target_columns)} classes")

    row_ids = sub_df["row_id"].values
    end_seconds = np.array([int(rid.rsplit("_", 1)[1]) for rid in row_ids])
    filenames = np.array([rid.rsplit("_", 1)[0] + ".ogg" for rid in row_ids])

    unique_files = list(dict.fromkeys(filenames))
    print(f"Unique soundscape files: {len(unique_files)}")

    existing_files = [f for f in unique_files if os.path.isfile(f"{TEST_DIR}/{f}")]
    if not existing_files:
        existing_files = [f for f in unique_files if os.path.isfile(f"{TRAIN_SC_DIR}/{f}")]
        soundscape_dir = TRAIN_SC_DIR
        print(f"No test .ogg found; using {len(existing_files)} train_soundscapes fallback files.")
    else:
        soundscape_dir = TEST_DIR
        if len(existing_files) < len(unique_files):
            print(f"  Available test files: {len(existing_files)}/{len(unique_files)}")
    unique_files = existing_files

    # Load ONNX models (skip if no files to process - Save & Run mode)
    sessions = []
    if len(unique_files) > 0:
        for fold_idx, onnx_path in enumerate(ONNX_PATHS):
            print(f"Loading ONNX model fold {fold_idx}: {onnx_path}")
            sessions.append(create_ort_session(onnx_path))
        input_name = sessions[0].get_inputs()[0].name
        print(f"  ONNX input name: {input_name}")
    else:
        print("No files to process, skipping model loading.")

    row_id_to_idx = {rid: i for i, rid in enumerate(row_ids)}
    predictions = sub_df[target_columns].values.astype(np.float32).copy()

    n_processed_files = 0
    n_processed_segments = 0
    t_inference_start = time.time()

    for file_idx, fname in enumerate(unique_files):
        filepath = f"{soundscape_dir}/{fname}"
        file_mask = filenames == fname
        file_end_seconds = end_seconds[file_mask]
        file_row_ids = row_ids[file_mask]

        audio, file_sr = sf.read(filepath, dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        segments = []
        for end_sec in file_end_seconds:
            start_sample = (end_sec - DURATION) * SR
            end_sample = end_sec * SR
            if start_sample < 0:
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[:max(end_sample, 0)]
                if len(available) > 0: seg[N_SAMPLES - len(available):] = available
            elif end_sample > len(audio):
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[min(start_sample, len(audio)):]
                if len(available) > 0: seg[:len(available)] = available
            else:
                seg = audio[start_sample:end_sample]
            segments.append(seg)

        mel_batch = compute_mel(torch.from_numpy(np.stack(segments)).float())

        clipwise_probs_sum = np.zeros((len(segments), NUM_CLASSES), dtype=np.float32)
        for session in sessions:
            for i in range(mel_batch.shape[0]):
                outputs = session.run(None, {input_name: mel_batch[i:i+1]})
                clipwise_probs_sum[i] += outputs[0][0]
        clipwise_probs = clipwise_probs_sum / len(sessions)

        for i, rid in enumerate(file_row_ids):
            predictions[row_id_to_idx[rid]] = clipwise_probs[i]

        n_processed_files += 1
        n_processed_segments += len(file_end_seconds)
        if n_processed_files % 100 == 0 or n_processed_files == len(unique_files):
            elapsed = time.time() - t_inference_start
            speed = n_processed_segments / elapsed if elapsed > 0 else 0
            print(f"  [{n_processed_files}/{len(unique_files)}] {n_processed_segments} segments, {elapsed:.1f}s, {speed:.1f} seg/s")

    sub_df[target_columns] = predictions
    sub_df.set_index("row_id").to_csv(SUBMISSION_FILE)

    t_total = time.time() - t_total_start
    print(f"\nInference complete: {n_processed_files} files, {n_processed_segments} segments, {t_total:.1f}s ({t_total/60:.1f}min)")
    print(f"Output: {SUBMISSION_FILE}")

if __name__ == "__main__":
    main()

## 4-Model Blend

In [ ]:
%%writefile blend.py

# ============================================================
# BirdCLEF 2026 | 4-Model Blend
# Reads lb872.csv, lb862.csv, effnetv2_b3.csv, eca_nfnet_l0.csv -> submission.csv
# ============================================================

import os
import numpy as np
import pandas as pd

DATA_ROOT  = "/kaggle/input/competitions/birdclef-2026"
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES    = list(sample_sub.columns[1:])
expected_ids = sample_sub["row_id"].tolist()

# Load all 4 model predictions
df_872   = pd.read_csv("lb872.csv", index_col="row_id")[SPECIES]
df_862   = pd.read_csv("lb862.csv", index_col="row_id")[SPECIES]
df_b3    = pd.read_csv("effnetv2_b3.csv", index_col="row_id")[SPECIES]
df_nfnet = pd.read_csv("eca_nfnet_l0.csv", index_col="row_id")[SPECIES]

preds_872   = df_872.values
preds_862   = df_862.values
preds_b3    = df_b3.values
preds_nfnet = df_nfnet.values

# Weighted blend: LB-proportional weights
# 0.886 + 0.885 + 0.872 + 0.862 = 3.505
W_B3    = 0.886 / 3.505
W_NFNET = 0.885 / 3.505
W_872   = 0.872 / 3.505
W_862   = 0.862 / 3.505

print(f"Blend weights: effnetv2_b3={W_B3:.3f}, eca_nfnet_l0={W_NFNET:.3f}, lb872={W_872:.3f}, lb862={W_862:.3f}")

blended = W_B3 * preds_b3 + W_NFNET * preds_nfnet + W_872 * preds_872 + W_862 * preds_862

# Post-processing on blended result
# File-Max Prior
FILE_MAX_WEIGHT = 0.05
SHARPEN_POWER = 1.5

# We need to group by file for post-processing
import re
row_pattern = re.compile(r"^(.*)_(\d+)$")
row_ids = df_b3.index.tolist()
stems = []
for rid in row_ids:
    m = row_pattern.match(str(rid))
    stems.append(m.group(1) if m else rid)

unique_stems = list(dict.fromkeys(stems))
stem_to_indices = {}
for i, s in enumerate(stems):
    stem_to_indices.setdefault(s, []).append(i)

for stem in unique_stems:
    indices = stem_to_indices[stem]
    n = len(indices)
    preds = blended[indices]

    # Confidence-Sharpened Smoothing
    if n > 4:
        weights = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
        pad = 2
        preds_sharp = np.power(preds, SHARPEN_POWER)
        padded = np.pad(preds_sharp, ((pad, pad), (0, 0)), mode='edge')
        smoothed = np.zeros_like(preds_sharp)
        for i_w in range(n):
            for j, w in enumerate(weights):
                smoothed[i_w] += w * padded[i_w + j]
        preds = np.power(smoothed, 1.0 / SHARPEN_POWER)
    elif n > 2:
        weights = np.array([0.20, 0.60, 0.20])
        pad = 1
        padded = np.pad(preds, ((pad, pad), (0, 0)), mode='edge')
        smoothed = np.zeros_like(preds)
        for i_w in range(n):
            for j, w in enumerate(weights):
                smoothed[i_w] += w * padded[i_w + j]
        preds = smoothed

    # File-Max Prior
    file_max = np.max(preds, axis=0, keepdims=True)
    preds = preds + FILE_MAX_WEIGHT * file_max

    blended[indices] = preds

# Build submission
submission = pd.DataFrame(blended, columns=SPECIES, index=df_b3.index)
submission = submission.loc[expected_ids].reset_index()
submission.to_csv("submission.csv", index=False)
print(f"Saved: submission.csv | {submission.shape}")

## Run All

In [ ]:
import time
t_start = time.time()

print("=" * 60)
print("MODEL 1: LB872 (Tony Li finetuned EfficientNet-B0)")
print("=" * 60)
!python lb872.py

print("=" * 60)
print("MODEL 2: LB862 (AidenSong baseline EfficientNet-B0)")
print("=" * 60)
!python lb862.py

print("=" * 60)
print("MODEL 3: effnetv2_b3 v2 ONNX 5-fold (LB 0.886)")
print("=" * 60)
!python effnetv2_b3.py

print("=" * 60)
print("MODEL 4: eca_nfnet_l0 ONNX 5-fold (LB 0.885)")
print("=" * 60)
!python eca_nfnet_l0.py

print("=" * 60)
print("BLEND")
print("=" * 60)
!python blend.py

t_total = time.time() - t_start
print(f"\nTotal pipeline time: {t_total:.1f}s ({t_total/60:.1f}min)")

In [ ]:
import os
assert os.path.isfile("submission.csv"), "ERROR: submission.csv was not created!"
import pandas as pd
df = pd.read_csv("submission.csv")
print(f"submission.csv: {df.shape}")
print(f"First 3 rows:\n{df.iloc[:3, :6].to_string()}")